# autoprof

> Functionality for running autoprof to measure the ICL in images of galaxy clusters.

In [5]:
# | default_exp autoprof

In [6]:
import glob
import numpy as np
from astropy.io import fits

import importlib.util
import sys
import os
import traceback
from pathlib import Path
from textwrap import dedent

import autoprof
from autoprof import Pipeline 


In [7]:
# | hide

In [8]:
# | export

def clean_and_process_images_for_autoprof(
    image_directory,
    mode="bulk",
    clusterid=None,
    bands=None,
    nan_value_replace='extreme',
    output_directory=None,
    boxsize=300
):
    """
    Cleans and processes images by replacing NaNs and saving to output directory.
    
    Parameters:
    - imdir (str): Directory where images are located.
    - mode (str): 'bulk' or 'single'. 'bulk' processes all matching images, 'single' targets one cluster.
    - clusterid (str): Required if mode is 'single'.
    - bands (list): List of bands to process. Default is ["H", "J", "Y", "HJY"].
    - nan_value_replace (str): 'extreme' to replace with 99, 'nanmin' to replace with np.nanmin.
    - output_directory (str or Path): Optional path to save cleaned images.
    - boxsize (int): Box size used in filename pattern.
    """
    if bands is None:
        bands = ["H", "J", "Y", "HJY"]

    for band in bands:
        if mode == "single":
            if not clusterid:
                raise ValueError("clusterid must be provided in single mode.")
            # pattern = f"{image_directory}{clusterid}_{band}_BKGSUB_boxsize_{boxsize}.fits"
            pattern = image_directory / f"EUC_NIR_W-STK_{band}-{clusterid}.fits"
            
        elif mode == "bulk":
            pattern = image_directory / f"EUC_NIR_W-STK_{band}-*.fits"
            
        else:
            raise ValueError("mode must be either 'bulk' or 'single'")

        image_files = sorted(glob.glob(str(pattern)))
        
        if not image_files:
            print(f"No files matched pattern: {pattern}")
            continue

        for image in image_files:
            image_path = Path(image)
            print(f"\nProcessing Image: {image_path.name}")
            
            new_clean_image_name = f"cleaned_{image_path.name}"

            with fits.open(image_path) as hdul:
                image_data = hdul[1].data.copy()
                image_header = hdul[1].header.copy()
                nan_mask = np.isnan(image_data)
                print(f"Number of NaN values: {nan_mask.sum()}")

                clean_image_data = image_data.copy()
                if nan_value_replace == 'extreme':
                    print("Replacing NaNs with 99.")
                    clean_image_data[nan_mask] = 99
                elif nan_value_replace == 'nanmin':
                    print("Replacing NaNs with nanmin.")
                    clean_image_data[nan_mask] = np.nanmin(clean_image_data)
                else:
                    raise ValueError("nan_value_replace must be 'extreme' or 'nanmin'.")

            save_dir = Path(output_directory) if output_directory else image_path.parent
            save_path = save_dir / new_clean_image_name
            save_dir.mkdir(parents=True, exist_ok=True)

            fits.writeto(save_path, clean_image_data, header=image_header, overwrite=True)
            print(f"Saved cleaned image to: {save_path}")

In [9]:
# | export

def run_autoprof(
    ids,
    image_files,
    mask_files,
    mode = "image list",
    unit_type= "intensity",
    gscale=0.4,
    pixelscale=0.3,
    zeropoint = 23.9,
    out_dir="./",
    config_name="basic_config.py",
    log_path="AutoProf.log",
    fourier_orders = None,
):
    """
    Running AutoProf on given image and mask files with a custom configuration.
    
    Parameters:
    - ids: List of strings for each image
    - image_files: List of paths to the image fits files
    - mask_files: List of paths to the mask fits files
    - mode : "image list" then ids, image and mask files needs to be arrays so it ca run for bulk.
    - unit_type: Either 'intensity' or 'mag'
    - gscale: Geometric scaling factor, default for noise diagnostics is 0.4 ~0.15dex, for clusters 0.1~0.05dex
    - out_dir: Output directory for saving results and logs
    - config_name: Name for the config Python file, so far I only magaed to run this software by calling its "basic_config" ...
      it crashes once this naming change, since it tries to access certain modules associated with it.

    - fourier_orders : To turn on extraction of Fourier coefficients, useful for lopsidedness analysis. I made it optional. Takes in a tuple (1,2,3)
    """
    os.makedirs(out_dir, exist_ok=True)
    
    config_file = os.path.join(out_dir, config_name)
    
    # Write the config file for AP
    with open(config_file, "w") as f:
        f.write(dedent(f"""\
import numpy as np
ap_process_mode = f"{mode}"
ap_name = {ids}
ap_image_file = {image_files}
ap_mask_file = {mask_files}
ap_saveto = "{out_dir}"
ap_pixscale = {pixelscale}
ap_zeropoint = {zeropoint}
ap_samplegeometricscale = {gscale}
ap_doplot = True
ap_extractfull = True
ap_fluxunits = "{unit_type}"
ap_isoclip = True
ap_isoclip_nsigma = 5
ap_ellipsefit = True
ap_fix_pa = False
ap_initial_pa = 45.0 
ap_fix_ellipticity = False
ap_initial_ellipticity = 0.3 
{"ap_iso_measurecoefs = " + str(fourier_orders) if fourier_orders else ""}
ap_new_pipeline_steps = [
    "mask segmentation map",
    "background",
    "psf",
    "center",
    "isophoteinit",
    "isophotefit",
    "isophoteextract",
    "writeprof",
]
"""))
    
    os.chdir(out_dir)   # Need to change the working directory for outputs, diagnostic plots will not be saved properly if this is not applied 


    try:
        import sys
        if config_name.replace(".py", "") in sys.modules:
            del sys.modules[config_name.replace(".py", "")]
        
        PIPELINE = Pipeline.Isophote_Pipeline(loggername=log_path)
        PIPELINE.Process_ConfigFile(config_name.replace(".py", ""))

        print(f"AutoProf completed successfully for {ids}!")
    except Exception as e:
        print(f"AutoProf failed for {ids}. Logging error...")

        error_log_file = Path(out_dir) / f"autoprof_error_{ids[0]}.log"
        with open(error_log_file, "w") as log:
            log.write(f"AutoProf failed for {ids}\n")
            log.write(f"Error type: {type(e).__name__}\n")
            log.write(f"Error message: {str(e)}\n")
            log.write("Full traceback:\n")
            log.write(traceback.format_exc())

        print(f"Error log saved to: {error_log_file}")


## The function that combines image prepping and running autoprof.
> Running the function below will create a "autoprof_results" folder in each cluster within cluster list provided.

> The prepped images will be contained as "cleaned" within the same autoprof_results folder.

> The autoprof will access the "cleaned" image from autoprof_results folder and mask from cluster_directory. The resultant diagnostic plots and profile files will be then stored in the created autoprof_results folder.

In [42]:
# | export

def process_cluster_with_autoprof(image_dir, cluster_directory, cluster_ids, filters):
    """
    image_dir: List of directories of the image.fits
    cluster_directory: directory if the cluster where is masks are located. The autoprof_results folder will be created within the respective cluster's directory.
    cluster_id: List of cluster IDs
    filters = list of band filters 
    """
    image_dir = Path(image_dir)
    
    for cluster_id in cluster_ids:
        
        cluster_directory = Path(cluster_directory) / cluster_id
        
        AP_results_dir = cluster_directory / "autoprof_results"
        AP_results_dir.mkdir(parents=True, exist_ok=True)
        
        for filter in filters:
            # Step 1: Clean and prepare image
            clean_and_process_images_for_autoprof(
                image_directory=image_dir,             
                output_directory=AP_results_dir,     
                mode='single',
                clusterid=cluster_id,
                bands=[filter]
            )
        
            cluster_band = f"{cluster_id}_{filter}"
            
            # Step 2: Run AutoProf
            run_autoprof(
                ids=[cluster_band],
                image_files=[str(AP_results_dir / f"cleaned_EUC_NIR_W-STK_{filter}-{cluster_id}.fits")],
                mask_files=[str(cluster_directory / f'{cluster_id}_{filter}_measurement_mask.fits')],
                out_dir=str(AP_results_dir),
                fourier_orders=(1, 4)
            )
            
            # Step 3: Keep only important output files
            files_to_keep = {
                "basic_config.py",
                f"{cluster_band}.prof",
                f"{cluster_band}.aux",
                f"Background_hist_{cluster_band}.jpg",
                f"mask_{cluster_band}.jpg",
                f"initialize_ellipse_{cluster_band}.jpg",
                f"photometry_{cluster_band}.jpg",
            }
            
            # Step 4: Clean up other files
            for f in AP_results_dir.iterdir():
                if f.is_file() and f.name not in files_to_keep:
                    if f"{filter}-{cluster_id}" in f.name or f"{cluster_id}_{filter}" in f.name:
                        try:
                            f.unlink()
                            print(f"Deleted: {f.name}")
                        except Exception as e:
                            print(f"Could not delete {f.name}: {e}")



# Example to run the image preparation and autoprof with existing mask file in output directory.

In [44]:
imdir = f"/home/ppztk1/euclid_data/Q1_R1_clusters_v0.7/tutku/"
outdir = f"/home/ppztk1/Erosita/Outputs_Clusters/"

clids = ["1eRASS J035423.6-475145"]
filters = ["H"] 

process_cluster_with_autoprof(image_dir=imdir, cluster_directory=outdir, cluster_ids=clid, filters=filters)
